In [ ]:
import requests
import numpy as np
from sgp4.api import Satrec
from astropy.time import Time
from astropy.coordinates import TEME, GCRS, CartesianRepresentation, CartesianDifferential
import astropy.units as u
from poliastro.bodies import Earth
from poliastro.twobody import Orbit
from poliastro.plotting import OrbitPlotter3D
import plotly.graph_objects as go
from datetime import datetime, timezone


In [ ]:
url = "https://celestrak.org/NORAD/elements/gp.php?CATNR=25544&FORMAT=TLE"
response = requests.get(url)
lines = response.text.strip().split('\n')

line1 = lines[1].strip()  # actual TLE line 1
line2 = lines[2].strip()  # actual TLE line 2
name  = lines[0].strip()  # "ISS (ZARYA)"

print(f" TLE fetched for: {name}")
print(f"Line 1: {line1}")
print(f"Line 2: {line2}")

In [ ]:
# Parse with sgp4
satellite = Satrec.twoline2rv(line1, line2)

# Get epoch
epoch = Time(satellite.jdsatepoch + satellite.jdsatepochF, format='jd', scale='utc')
print(f"Epoch: {epoch.iso}")

# Propagate at epoch (t=0) → position & velocity in TEME frame
e, r_teme, v_teme = satellite.sgp4(satellite.jdsatepoch, satellite.jdsatepochF)

# Wrap in astropy TEME
r_teme_coord = CartesianRepresentation(r_teme * u.km)
v_teme_coord = CartesianDifferential(v_teme * u.km / u.s)
teme = TEME(r_teme_coord.with_differentials(v_teme_coord), obstime=epoch)

# Convert TEME → GCRS
gcrs = teme.transform_to(GCRS(obstime=epoch))
r_gcrs = gcrs.cartesian.xyz.to(u.km)
v_gcrs = gcrs.cartesian.differentials['s'].d_xyz.to(u.km / u.s)

# Build poliastro orbit
ss = Orbit.from_vectors(Earth, r_gcrs, v_gcrs, epoch=epoch)

print(f" Orbit created!")
print(f"   Altitude : {(ss.a - Earth.R).to(u.km):.1f}")
print(f"   Inclination: {ss.inc.to(u.deg):.2f}")
print(f"   Period   : {ss.period.to(u.min):.2f}")

In [ ]:
num_steps = 500
period_s = ss.period.to(u.s).value
time_deltas = np.linspace(0, period_s, num_steps) * u.s

positions = np.array([
    ss.propagate(dt).r.to(u.km).value
    for dt in time_deltas
])

print(f" Propagated {num_steps} points over one full orbit")

In [ ]:
now = Time(datetime.now(timezone.utc))
dt_now = (now - epoch).to(u.s)
ss_now = ss.propagate(dt_now)

iss_pos = ss_now.r.to(u.km).value
print(f" Current ISS position (GCRS, km):")
print(f"   X: {iss_pos[0]:.1f}  Y: {iss_pos[1]:.1f}  Z: {iss_pos[2]:.1f}")

In [ ]:
# Earth sphere
R_earth = 6371
u_s = np.linspace(0, 2 * np.pi, 80)
v_s = np.linspace(0, np.pi, 80)
x_e = R_earth * np.outer(np.cos(u_s), np.sin(v_s))
y_e = R_earth * np.outer(np.sin(u_s), np.sin(v_s))
z_e = R_earth * np.outer(np.ones(80), np.cos(v_s))

earth = go.Surface(
    x=x_e, y=y_e, z=z_e,
    colorscale=[[0, '#1a6fa8'], [1, '#1a6fa8']],
    showscale=False,
    opacity=0.45,
    name='Earth'
)

orbit_line = go.Scatter3d(
    x=positions[:, 0],
    y=positions[:, 1],
    z=positions[:, 2],
    mode='lines',
    line=dict(color='cyan', width=2),
    name='ISS Orbit'
)

iss_dot = go.Scatter3d(
    x=[iss_pos[0]],
    y=[iss_pos[1]],
    z=[iss_pos[2]],
    mode='markers+text',
    marker=dict(size=6, color='red'),
    text=['ISS'],
    textfont=dict(color='white', size=12),
    name='ISS Now'
)

fig = go.Figure(data=[earth, orbit_line, iss_dot])
fig.update_layout(
    title='🛰️ ISS Live Orbit',
    scene=dict(
    xaxis=dict(
        title='X (km)', color='white',
        showgrid=False, zeroline=True,
        showbackground=False,
        tickvals=[-6000, 0], ticktext=['-6000', '0']
    ),
    yaxis=dict(
        title='Y (km)', color='white',
        showgrid=False, zeroline=True,
        showbackground=False,
        tickvals=[-6000, 0, 6000], ticktext=['-6000', '0', '6000']
    ),
    zaxis=dict(
        title='Z (km)', color='white',
        showgrid=False, zeroline=True,
        showbackground=False,
        tickvals=[0, 6000], ticktext=['0', '6000']
    ),
    bgcolor='black',
    aspectmode='data'
),
    paper_bgcolor='black',
    font=dict(color='white'),
    legend=dict(bgcolor='black')
)

fig.show()

In [ ]:
# Extract orbital elements
inc  = ss.inc.to(u.rad).value        # inclination in radians
raan = ss.raan.to(u.rad).value       # RAAN in radians
argp = ss.argp.to(u.rad).value       # argument of perigee in radians
ecc  = ss.ecc.value                  # eccentricity (dimensionless)

print(f"Inclination:          {np.degrees(inc):.2f}°")
print(f"RAAN:                 {np.degrees(raan):.2f}°")
print(f"Argument of Perigee:  {np.degrees(argp):.2f}°")
print(f"Eccentricity:         {ecc:.6f}")

In [ ]:
# Equatorial plane as a flat square
plane_size = 8000  # km, slightly bigger than Earth

# Four corners of the square in the XY plane (Z=0)
sq = np.linspace(-plane_size, plane_size, 2)
xx, yy = np.meshgrid(sq, sq)
zz = np.zeros_like(xx)

equatorial_plane = go.Surface(
    x=xx, y=yy, z=zz,
    colorscale=[[0, 'rgba(225, 225, 255,0.4)'], [1, 'rgba(225, 225, 255,0.4)']],
    showscale=False,
    opacity=1,
    name='Equatorial Plane',
    hovertemplate='Equatorial Plane<extra></extra>'
)

fig_test = go.Figure(data=[earth, orbit_line, iss_dot, equatorial_plane])
fig_test.update_layout(
    title='🛰️ ISS Orbit + Equatorial Plane',
    scene=dict(
    xaxis=dict(
        title='X (km)', color='white',
        showgrid=False, zeroline=True,
        showbackground=False,
        tickvals=[-6000, 0], ticktext=['-6000', '0']
    ),
    yaxis=dict(
        title='Y (km)', color='white',
        showgrid=False, zeroline=True,
        showbackground=False,
        tickvals=[-6000, 0, 6000], ticktext=['-6000', '0', '6000']
    ),
    zaxis=dict(
        title='Z (km)', color='white',
        showgrid=False, zeroline=True,
        showbackground=False,
        tickvals=[0, 6000], ticktext=['0', '6000']
    ),
        bgcolor='black',
        aspectmode='data'
    ),
    paper_bgcolor='black',
    font=dict(color='white'),
    legend=dict(bgcolor='black')
)
fig_test.show()


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go

def make_fig(arc_r):
    # Inclination arc
    ref_angle = np.arctan2(-h_proj[1], -h_proj[0])
    arc_angles = np.linspace(0, inc, 50)

    arc_x = arc_r * np.cos(arc_angles) * np.cos(ref_angle)
    arc_y = arc_r * np.cos(arc_angles) * np.sin(ref_angle)
    arc_z = arc_r * np.sin(arc_angles)

    inc_arc = go.Scatter3d(
        x=arc_x, y=arc_y, z=arc_z,
        mode='lines',
        line=dict(color='yellow', width=4),
        name=f'Inclination ({np.degrees(inc):.1f}°)',
        hovertemplate=f'Inclination: {np.degrees(inc):.1f}°<extra></extra>'
    )

    mid = len(arc_angles) // 2
    inc_label = go.Scatter3d(
        x=[arc_x[mid] * 1.4],
        y=[arc_y[mid] * 1.4],
        z=[arc_z[mid] * 1.4],
        mode='text',
        text=[f'i = {np.degrees(inc):.1f}°'],
        textfont=dict(color='yellow', size=11),
        showlegend=False,
        hoverinfo='skip'
    )

    fig = go.Figure(data=[earth, orbit_line, iss_dot, equatorial_plane, inc_arc, inc_label, reference_line])
    fig.update_layout(
        title=f'🛰️ ISS Orbit + Inclination Arc (r = {arc_r:.0f} km)',
        scene=dict(
            xaxis=dict(title='X (km)', color='white', showgrid=False, zeroline=False, showbackground=False,
                       tickvals=[-6000, 0, 6000], ticktext=['-6000', '0', '6000']),
            yaxis=dict(title='Y (km)', color='white', showgrid=False, zeroline=False, showbackground=False,
                       tickvals=[-6000, 0, 6000], ticktext=['-6000', '0', '6000']),
            zaxis=dict(title='Z (km)', color='white', showgrid=False, zeroline=False, showbackground=False,
                       tickvals=[-6000, 0, 6000], ticktext=['-6000', '0', '6000']),
            bgcolor='black',
            aspectmode='data'
        ),
        paper_bgcolor='black',
        font=dict(color='white'),
        legend=dict(bgcolor='black')
    )
    return fig



In [ ]:
import ipywidgets as widgets
from IPython.display import display

# ── 1. h_proj ────────────────────────────────────────────────────
h_vec = np.array([
    np.sin(inc) * np.sin(raan),
   -np.sin(inc) * np.cos(raan),
    np.cos(inc)
])
h_proj = np.array([h_vec[0], h_vec[1], 0])
h_proj = h_proj / np.linalg.norm(h_proj)

# ── 2. Reference line ─────────────────────────────────────────────
ref_len = 8000
reference_line = go.Scatter3d(
    x=[0, -h_proj[0] * ref_len],
    y=[0, -h_proj[1] * ref_len],
    z=[0, 0],
    mode='lines+text',
    line=dict(color='white', width=3),
    text=['', 'ref'],
    textfont=dict(color='white', size=10),
    textposition='top center',
    name='Reference Direction',
    hovertemplate='Normal to orbital plane (projected)<extra></extra>'
)

# ── 3. Arc helper ─────────────────────────────────────────────────
ref_angle = np.arctan2(-h_proj[1], -h_proj[0])
arc_angles = np.linspace(0, inc, 50)
mid = len(arc_angles) // 2

def compute_arc(arc_r):
    ax = arc_r * np.cos(arc_angles) * np.cos(ref_angle)
    ay = arc_r * np.cos(arc_angles) * np.sin(ref_angle)
    az = arc_r * np.sin(arc_angles)
    return ax, ay, az

# ── 4. Initial arc traces ─────────────────────────────────────────
arc_r_init = 3185
ax0, ay0, az0 = compute_arc(arc_r_init)

inc_arc = go.Scatter3d(
    x=ax0, y=ay0, z=az0,
    mode='lines',
    line=dict(color='yellow', width=4),
    name=f'Inclination ({np.degrees(inc):.1f}°)',
    hovertemplate=f'Inclination: {np.degrees(inc):.1f}°<extra></extra>'
)

inc_label = go.Scatter3d(
    x=[ax0[mid] * 1.4],
    y=[ay0[mid] * 1.4],
    z=[az0[mid] * 1.4],
    mode='text',
    text=[f'i = {np.degrees(inc):.1f}°'],
    textfont=dict(color='yellow', size=11),
    showlegend=False,
    hoverinfo='skip'
)

# ── 5. FigureWidget ───────────────────────────────────────────────
fw = go.FigureWidget(data=[earth, orbit_line, iss_dot, equatorial_plane, inc_arc, inc_label, reference_line])
fw.update_layout(
    title='🛰️ ISS Orbit + Equatorial Plane + Inclination',
    scene=dict(
        xaxis=dict(title='X (km)', color='white', showgrid=False, zeroline=False, showbackground=False,
                   tickvals=[0, 6000], ticktext=['0', '6000']),
        yaxis=dict(title='Y (km)', color='white', showgrid=False, zeroline=False, showbackground=False,
                   tickvals=[-6000, 0], ticktext=['-6000', '0']),
        zaxis=dict(title='Z (km)', color='white', showgrid=False, zeroline=False, showbackground=False,
                   tickvals=[-6000, 0, 6000], ticktext=['-6000', '0', '6000']),
        bgcolor='black',
        aspectmode='data'
    ),
    paper_bgcolor='black',
    font=dict(color='white'),
    legend=dict(bgcolor='black')
)

# ── 6. Slider ─────────────────────────────────────────────────────
def on_slider(change):
    ax, ay, az = compute_arc(change['new'])
    with fw.batch_update():
        fw.data[4].x = tuple(ax)
        fw.data[4].y = tuple(ay)
        fw.data[4].z = tuple(az)
        fw.data[5].x = (ax[mid] * 1.4,)
        fw.data[5].y = (ay[mid] * 1.4,)
        fw.data[5].z = (az[mid] * 1.4,)

slider = widgets.IntSlider(
    value=arc_r_init,
    min=0,
    max=6371,
    step=50,
    description='Arc r (km):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

slider.observe(on_slider, names='value')
display(slider, fw)
fw.write_html("iss_orbit.html")

In [ ]:
# Orbital plane as a tilted square
# Build it in the orbital frame then rotate it into the inertial frame

plane_size = 8000

# Square in local orbital plane coordinates (before rotation)
sq = np.linspace(-plane_size, plane_size, 2)
xx_loc, yy_loc = np.meshgrid(sq, sq)
zz_loc = np.zeros_like(xx_loc)

# Rotation matrix: first rotate by RAAN around Z, then by inc around X
# This transforms local orbital plane coords into inertial (GCRS) frame
def R_z(angle):
    return np.array([
        [np.cos(angle), -np.sin(angle), 0],
        [np.sin(angle),  np.cos(angle), 0],
        [0,              0,             1]
    ])

def R_x(angle):
    return np.array([
        [1, 0,           0          ],
        [0, np.cos(angle), -np.sin(angle)],
        [0, np.sin(angle),  np.cos(angle)]
    ])

# Combined rotation: Rz(RAAN) * Rx(inc)
R = R_z(raan) @ R_x(inc)

# Apply rotation to every point on the grid
shape = xx_loc.shape
pts = np.stack([xx_loc.ravel(), yy_loc.ravel(), zz_loc.ravel()])  # 3 x N
pts_rot = R @ pts  # rotate

xx_orb = pts_rot[0].reshape(shape)
yy_orb = pts_rot[1].reshape(shape)
zz_orb = pts_rot[2].reshape(shape)

orbital_plane = go.Mesh3d(
    x=[xx_orb[0,0], xx_orb[0,1], xx_orb[1,0], xx_orb[1,1]],
    y=[yy_orb[0,0], yy_orb[0,1], yy_orb[1,0], yy_orb[1,1]],
    z=[zz_orb[0,0], zz_orb[0,1], zz_orb[1,0], zz_orb[1,1]],
    i=[0, 1],
    j=[1, 2],
    k=[2, 3],
    color='orange',
    opacity=0.4,
    flatshading=True,
    lighting=dict(ambient=1, diffuse=0, specular=0),
    name='Orbital Plane',
    hovertemplate='Orbital Plane<extra></extra>'
)

fw2 = go.FigureWidget(data=[earth, orbit_line, iss_dot, equatorial_plane, orbital_plane, inc_arc, inc_label, reference_line])
fw2.update_layout(
    title='🛰️ ISS Orbit + Equatorial + Orbital Plane',
    scene=dict(
        xaxis=dict(title='X (km)', color='white', showgrid=False, zeroline=False, showbackground=False,
                   tickvals=[-6000, 0], ticktext=['-6000', '0']),
        yaxis=dict(title='Y (km)', color='white', showgrid=False, zeroline=False, showbackground=False,
                   tickvals=[0, 6000], ticktext=['0', '6000']),
        zaxis=dict(title='Z (km)', color='white', showgrid=False, zeroline=False, showbackground=False,
                   tickvals=[-6000, 0, 6000], ticktext=['-6000', '0', '6000']),
        bgcolor='black',
        aspectmode='data'
    ),
    paper_bgcolor='black',
    font=dict(color='white'),
    legend=dict(bgcolor='black')
)

fw2.write_html("iss_orbit_orbital_plane.html")
fw2.show()
print("✅ Orbital plane added!")